# 3.0.2 - Phân chia tập dữ liệu Train / Validation / Test theo thời gian (Time-Based Split)

### 🎯 Mục tiêu:
Thêm 1 cột `Split` để phân chia tập dữ liệu thành 3 phần theo thứ tự thời gian tăng dần:
- **`1` = Train (Huấn luyện):** 70% dữ liệu lịch sử đầu tiên
- **`2` = Validation (Kiểm định / Tinh chỉnh siêu tham số):** 15% dữ liệu tiếp theo
- **`3` = Test (Kiểm thử Out-of-Sample cuối cùng):** 15% dữ liệu mới nhất

### 🛡️ Nguyên tắc chống Data Leakage:
- Tuyệt đối chia tuần tự theo trục thời gian (Time-Based Split), không xáo trộn ngẫu nhiên (No Random Shuffle).
- Thứ tự: $\text{Train} < \text{Validation} < \text{Test}$.

---
- **Dữ liệu đầu vào:** `dataset/3.0.1_drop_missing.csv`
- **Dữ liệu đầu ra:** `dataset/3.0.2_split.csv`

In [ ]:
import os
import numpy as np
import pandas as pd

# 1. Đọc dữ liệu từ phiên bản 3.0.1_drop_missing.csv
dataset_dir = 'dataset' if os.path.exists('dataset') else ('../dataset' if os.path.exists('../dataset') else '.')
input_path = os.path.join(dataset_dir, '3.0.1_drop_missing.csv')
print(f"Đọc dữ liệu từ: {input_path}")
df = pd.read_csv(input_path)

# Đảm bảo dữ liệu sắp xếp đúng theo thứ tự thời gian
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)
print(f"Kích thước dữ liệu sạch: {df.shape[0]} dòng, {df.shape[1]} cột.")
display(df.head(3))

In [ ]:
# 2. Phân chia tỷ lệ 70% Train (1) - 15% Validation (2) - 15% Test (3)
n_total = len(df)
n_train = int(n_total * 0.70)
n_val = int(n_total * 0.15)
n_test = n_total - n_train - n_val

# Gán nhãn cột Split (1: Train, 2: Validation, 3: Test)
df['Split'] = [1] * n_train + [2] * n_val + [3] * n_test

# Thống kê chi tiết từng tập
train_data = df[df['Split'] == 1]
val_data = df[df['Split'] == 2]
test_data = df[df['Split'] == 3]

print("=== KẾT QUẢ PHÂN CHIA TẬP DỮ LIỆU ===")
print(f"• 1 = Train      : {len(train_data):4d} dòng ({len(train_data)/n_total*100:.1f}%) | Từ {train_data['Date'].iloc[0].date()} đến {train_data['Date'].iloc[-1].date()}")
print(f"• 2 = Validation : {len(val_data):4d} dòng ({len(val_data)/n_total*100:.1f}%) | Từ {val_data['Date'].iloc[0].date()} đến {val_data['Date'].iloc[-1].date()}")
print(f"• 3 = Test       : {len(test_data):4d} dòng ({len(test_data)/n_total*100:.1f}%) | Từ {test_data['Date'].iloc[0].date()} đến {test_data['Date'].iloc[-1].date()}")

In [ ]:
# 3. Kiểm tra phân phối Volatility_Regime trên từng tập
print("Phân phối trạng thái thị trường (Volatility_Regime) trên từng tập:\n")
split_regime = pd.crosstab(df['Split'].map({1: '1_Train', 2: '2_Validation', 3: '3_Test'}), df['Volatility_Regime'], margins=True)
display(split_regime)

In [ ]:
# 4. Xuất file kết quả phiên bản 3.0.2_split.csv
output_path = os.path.join(dataset_dir, '3.0.2_split.csv')
df.to_csv(output_path, index=False)

print(f"✅ ĐÃ XUẤT THÀNH CÔNG TẠI: {output_path}")
print(f"Kích thước Dataset mới: {df.shape[0]} dòng, {df.shape[1]} cột (đã thêm cột Split).")
display(df[['Date', 'Close', 'Return_1D', 'Volatility_Regime', 'Split']].head(3))
display(df[['Date', 'Close', 'Return_1D', 'Volatility_Regime', 'Split']].tail(3))